# SparseVol3D — Experiment Notebook

**Sparse Axial Supervision for 3D Medical Image Segmentation with Volumetric Interpolation Consistency**

This notebook walks through:
1. Setup and data check
2. Visualise a CT volume + segmentation
3. Configure and train a single experiment
4. Plot training curves
5. Evaluate on validation set
6. Visualise predictions vs ground truth

## 1. Setup

In [ ]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
import torch

# Make sure project root is on the path
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
print(f'Root     : {PROJECT_ROOT}')

## 2. Configuration

Change these values to run different experiments.

In [ ]:
# ── Experiment settings ────────────────────────────────────────────────────
DATA_DIR    = 'data/kits19/data'   # path to KiTS19 cases
OUTPUT_DIR  = 'outputs/notebook_run'

LABEL_STRIDE = 5      # 1=dense, 5=every 5th slice (20%), 10=every 10th (10%)
LAMBDA_VIC   = 0.1    # VIC loss weight; set 0.0 to disable
EPOCHS       = 50     # increase to 100 for full training
BATCH_SIZE   = 2
BASE_CHANNELS = 32    # reduce to 16 to save VRAM

# ── Auto debug mode on CPU ─────────────────────────────────────────────────
DEBUG = not torch.cuda.is_available()
if DEBUG:
    print('No GPU found — enabling debug mode (tiny model, 3 epochs)')
    EPOCHS, BATCH_SIZE, BASE_CHANNELS = 3, 1, 8

print(f'label_stride = {LABEL_STRIDE}  ({100 // LABEL_STRIDE}% annotation budget)')
print(f'lambda_vic   = {LAMBDA_VIC}')
print(f'epochs       = {EPOCHS}')

## 3. Data

Generate synthetic data if KiTS19 is not downloaded yet.

In [ ]:
import glob

cases = sorted(glob.glob(os.path.join(DATA_DIR, 'case_*')))
print(f'Found {len(cases)} cases in {DATA_DIR}')

if len(cases) == 0:
    print('No cases found — generating 10 synthetic cases...')
    os.system(f'python scripts/make_synthetic_data.py --n_cases 10 --data_dir {DATA_DIR}')
    cases = sorted(glob.glob(os.path.join(DATA_DIR, 'case_*')))
    print(f'Generated {len(cases)} cases')

In [ ]:
# Generate splits if not present
splits_path = 'outputs/splits.json'
if not os.path.exists(splits_path):
    print('Generating splits...')
    os.system(f'python scripts/prepare_splits.py --data_dir {DATA_DIR}')

with open(splits_path) as f:
    splits = json.load(f)

print(f"Train: {len(splits['train'])} cases")
print(f"Val  : {len(splits['val'])} cases")
print(f"Test : {len(splits['test'])} cases")

### Visualise a sample volume

In [ ]:
import nibabel as nib

# Load the first training case
case_id = splits['train'][0]
case_dir = os.path.join(DATA_DIR, f'case_{case_id:05d}')
img = nib.load(f'{case_dir}/imaging.nii.gz').get_fdata(dtype='float32')
seg = nib.load(f'{case_dir}/segmentation.nii.gz').get_fdata().astype(int)

print(f'Volume shape : {img.shape}')
print(f'Seg classes  : {np.unique(seg)}')

# Pick 3 evenly-spaced axial slices
D = img.shape[0]
slices = [D // 4, D // 2, 3 * D // 4]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle(f'Case {case_id:05d}  |  shape {img.shape}', fontsize=13)

for col, z in enumerate(slices):
    # CT image
    axes[0, col].imshow(img[z], cmap='gray', vmin=-200, vmax=300)
    axes[0, col].set_title(f'CT slice z={z}')
    axes[0, col].axis('off')

    # Segmentation overlay
    axes[1, col].imshow(img[z], cmap='gray', vmin=-200, vmax=300)
    masked_kidney = np.ma.masked_where(seg[z] != 1, seg[z])
    masked_tumor  = np.ma.masked_where(seg[z] != 2, seg[z])
    axes[1, col].imshow(masked_kidney, cmap='Blues',  alpha=0.5, vmin=0, vmax=2)
    axes[1, col].imshow(masked_tumor,  cmap='Reds',   alpha=0.6, vmin=0, vmax=2)
    axes[1, col].set_title(f'Seg z={z}  (blue=kidney, red=tumor)')
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig('outputs/sample_volume.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved to outputs/sample_volume.png')

## 4. Train

In [ ]:
import random
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

from data import KiTS19Dataset
from models import UNet3D
from losses import combined_loss
from utils import compute_dice

# Reproducibility
random.seed(42); np.random.seed(42); torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(OUTPUT_DIR, exist_ok=True)

patch_size = (16, 64, 64) if DEBUG else (64, 128, 128)

train_ds = KiTS19Dataset(DATA_DIR, splits['train'], patch_size, LABEL_STRIDE, mode='train')
val_ds   = KiTS19Dataset(DATA_DIR, splits['val'],   patch_size, label_stride=1, mode='val')

nw = 0 if DEBUG else 4
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=nw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=nw)

model = UNet3D(in_channels=1, num_classes=3, base_channels=BASE_CHANNELS).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model params: {n_params/1e6:.2f} M  |  device: {device}')

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
cuda_ok   = torch.cuda.is_available()
scaler    = GradScaler('cuda') if cuda_ok else None

In [ ]:
train_losses, val_kidney, val_tumor = [], [], []

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    epoch_loss = 0.0
    for img, seg, mask in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False):
        img, seg, mask = img.to(device), seg.to(device), mask.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=scaler is not None):
            logits = model(img)
            loss   = combined_loss(logits, seg, mask, lambda_vic=LAMBDA_VIC)
        if scaler:
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        else:
            loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    train_losses.append(epoch_loss / len(train_loader))

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    k_dice, t_dice = [], []
    with torch.no_grad():
        for img, seg, _ in val_loader:
            logits = model(img.to(device))
            pred   = logits.argmax(dim=1)
            for b in range(pred.shape[0]):
                d = compute_dice(pred[b], seg[b].to(device))
                k_dice.append(d[1]); t_dice.append(d[2])
    val_kidney.append(np.mean(k_dice))
    val_tumor.append(np.mean(t_dice))

    if epoch % 5 == 0 or epoch == EPOCHS:
        print(f'Epoch {epoch:3d} | loss {train_losses[-1]:.4f} | '
              f'kidney {val_kidney[-1]:.4f} | tumor {val_tumor[-1]:.4f}')

# Save checkpoint
torch.save({'model': model.state_dict(),
            'epoch': EPOCHS,
            'config': {'num_classes': 3, 'base_channels': BASE_CHANNELS,
                       'patch_size': list(patch_size), 'label_stride': LABEL_STRIDE}},
           f'{OUTPUT_DIR}/model.pt')
print(f'Checkpoint saved to {OUTPUT_DIR}/model.pt')

## 5. Training Curves

In [ ]:
epochs = list(range(1, len(train_losses) + 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs, train_losses, color='steelblue', linewidth=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(alpha=0.3)

ax2.plot(epochs, val_kidney, label='Kidney', color='royalblue', linewidth=2)
ax2.plot(epochs, val_tumor,  label='Tumor',  color='tomato',    linewidth=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Dice')
ax2.set_title(f'Validation Dice  (stride={LABEL_STRIDE}, VIC={LAMBDA_VIC})')
ax2.legend(); ax2.grid(alpha=0.3); ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Final   | Kidney {val_kidney[-1]:.4f} | Tumor {val_tumor[-1]:.4f}')
print(f'Best    | Kidney {max(val_kidney):.4f} | Tumor {max(val_tumor):.4f}')

## 6. Visualise Predictions

In [ ]:
import random
import nibabel as nib
from data.kits_dataset import _pad3d

# ── Pick 3 random val cases ───────────────────────────────────────────────────
chosen   = random.sample(splits['val'], min(3, len(splits['val'])))
n_cases  = len(chosen)
n_slices = 3    # axial slices per case

fig, axes = plt.subplots(n_cases * 2, n_slices, figsize=(n_slices * 4, n_cases * 5))
fig.suptitle('Predictions on random val cases\n(blue = kidney, red = tumor)', fontsize=13)

if n_cases == 1:
    axes = axes.reshape(2, n_slices)

model.eval()
for row_pair, case_id in enumerate(chosen):
    case_dir = os.path.join(DATA_DIR, f'case_{case_id:05d}')
    img_raw  = nib.load(f'{case_dir}/imaging.nii.gz').get_fdata(dtype='float32')
    seg_raw  = nib.load(f'{case_dir}/segmentation.nii.gz').get_fdata().astype(int)

    # Preprocess + center crop to patch
    img_pp = np.clip(img_raw, -200, 300)
    img_pp = ((img_pp + 200.0) / 500.0).astype(np.float32)
    D, H, W = img_pp.shape
    pd, ph, pw = patch_size
    d0 = max(0, (D - pd) // 2)
    h0 = max(0, (H - ph) // 2)
    w0 = max(0, (W - pw) // 2)

    img_p = _pad3d(img_pp[d0:d0+pd, h0:h0+ph, w0:w0+pw], patch_size)
    seg_p = _pad3d(seg_raw[d0:d0+pd, h0:h0+ph, w0:w0+pw], patch_size)

    with torch.no_grad():
        t    = torch.from_numpy(img_p[None, None]).float().to(device)
        pred = model(t).argmax(dim=1).cpu().numpy()[0]

    show_z   = [pd // 4, pd // 2, 3 * pd // 4]
    row_gt   = row_pair * 2
    row_pred = row_pair * 2 + 1

    for col, z in enumerate(show_z):
        ct = img_p[z] * 500 - 200    # undo normalisation for display

        for row, overlay in [(row_gt, seg_p), (row_pred, pred)]:
            axes[row, col].imshow(ct, cmap='gray', vmin=-200, vmax=300)
            axes[row, col].imshow(np.ma.masked_where(overlay[z] != 1, overlay[z]),
                                  cmap='Blues', alpha=0.5, vmin=0, vmax=2)
            axes[row, col].imshow(np.ma.masked_where(overlay[z] != 2, overlay[z]),
                                  cmap='Reds',  alpha=0.6, vmin=0, vmax=2)
            label = 'GT' if row == row_gt else 'Pred'
            axes[row, col].set_title(f'{label}  case_{case_id:05d}  z={z}', fontsize=8)
            axes[row, col].axis('off')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/visual_predictions.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved to {OUTPUT_DIR}/visual_predictions.png')

## 7. Summary

In [ ]:
print('=' * 50)
print('  EXPERIMENT SUMMARY')
print('=' * 50)
print(f'  label_stride : {LABEL_STRIDE}  ({100 // LABEL_STRIDE}% annotation)')
print(f'  lambda_vic   : {LAMBDA_VIC}')
print(f'  epochs       : {EPOCHS}')
print(f'  device       : {device}')
print()
print(f'  Best Kidney Dice : {max(val_kidney):.4f}')
print(f'  Best Tumor  Dice : {max(val_tumor):.4f}')
print(f'  Best Mean   Dice : {(max(val_kidney) + max(val_tumor)) / 2:.4f}')
print('=' * 50)
print(f'  Outputs saved to: {OUTPUT_DIR}/')